# BBOX Model Version 1

## Import Libraries

In [12]:
# ==== Core utilities ====
from pathlib import Path
import os, sys, json, math, random, shutil, glob, itertools, warnings
from collections import defaultdict, Counter

# ==== Scientific stack ====
import numpy as np
import pandas as pd
import random
import tempfile

# ==== Plotting ====
import matplotlib.pyplot as plt

# ==== Image I/O ====
from PIL import Image

# ==== Splitting (parent-aware via grouping + stratification) ====
from sklearn.model_selection import StratifiedKFold




In [13]:
%pip install -q ultralytics
from ultralytics import YOLO

Note: you may need to restart the kernel to use updated packages.


## Import Data

In [14]:
BASE_DIR = Path("/kaggle/input/bboxdataset")
PATCHES_DIR = BASE_DIR / "patches"
PATCHES_BBOX_JSON = BASE_DIR / "patches_bbox.json"

## Detector choice and evaluation protocol

We adopt YOLOv8 (Ultralytics) as the baseline detector. It offers a strong speed/accuracy trade-off, a stable training recipe on small datasets, native export of mAP metrics and PR curves, and a simple Python API. We evaluate with mAP@0.5 (Object detection is evaluated using mean Average Precision (mAP), where precision/recall are computed at different IoU thresholds.) and mAP@[0.5:0.95] (COCO protocol, stricter localization). We also inspect per-class AP, PR curves, and qualitative overlays. To prevent leakage, the train/val split is parent-aware: patches derived from the same parent image_id never cross splits.

In [15]:
# Repro seeds (YOLO uses its own too, but keep NumPy/Python stable)
SEED = 42
random.seed(SEED); np.random.seed(SEED)
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

In [16]:
# ---- Workspace persistente ----
WORKDIR = Path("/kaggle/working/yolo_patches")
WORKDIR.mkdir(parents=True, exist_ok=True)
print("Workspace:", WORKDIR)

Workspace: /kaggle/working/yolo_patches


## Load patch annotations and summarize schema

Load the custom patch JSON (flat list under patches) and construct a dataframe with file_name, parent image_id, category_id, and [x, y, w, h] bounding boxes in pixels. We then compute basic counts (number of patches, categories represented) and verify that all referenced images exist on disk.

In [17]:
with open(PATCHES_BBOX_JSON, "r", encoding="utf-8") as f:
    raw = json.load(f)

patch_records = raw.get("patches", [])
assert isinstance(patch_records, list) and len(patch_records) > 0, "No patches found in JSON."

In [18]:
# Build a DataFrame
df_p = pd.DataFrame.from_records(patch_records)
expected_cols = {"file_name","image_id","category_id","bbox"}
missing = expected_cols - set(df_p.columns)
assert not missing, f"JSON missing required keys: {missing}"

In [19]:
# Normalizza i nomi file e risolvi i path assoluti a prova di sottocartelle/estensioni
valid_ext = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}

# 1) tieni solo il nome (niente prefix tipo 'patches/...')
df_p["file_name"] = df_p["file_name"].apply(lambda x: Path(x).name)
df_p["fname_lower"] = df_p["file_name"].str.lower()

# 2) indicizza ricorsivamente tutte le immagini dentro il dataset
index = {}  # filename_lower -> Path
for p in BASE_DIR.rglob("*"):
    if p.is_file() and p.suffix.lower() in valid_ext:
        index[p.name.lower()] = p

# 3) mappa diretta (case-insensitive)
df_p["abs_path"] = df_p["fname_lower"].map(index)

# 4) fallback: se l'estensione nel JSON non coincide, prova tutte le estensioni comuni
def alt_lookup(name_lower: str):
    stem = Path(name_lower).stem
    for ext in valid_ext:
        cand = stem + ext
        if cand in index:
            return index[cand]
    return None

need_alt = df_p["abs_path"].isna()
df_p.loc[need_alt, "abs_path"] = df_p.loc[need_alt, "fname_lower"].map(alt_lookup)

# 5) debug
missing = df_p["abs_path"].isna().sum()
print(f"Righe senza file risolto: {missing} su {len(df_p)}")
print("Esempi risolti:", df_p.loc[df_p["abs_path"].notna(), "abs_path"].head().tolist())
if missing:
    print("Esempi mancanti:", df_p.loc[df_p["abs_path"].isna(), "file_name"].head(10).tolist())


Righe senza file risolto: 0 su 6000
Esempi risolti: [PosixPath('/kaggle/input/bboxdataset/patches/patches/patch_000000.png'), PosixPath('/kaggle/input/bboxdataset/patches/patches/patch_000001.png'), PosixPath('/kaggle/input/bboxdataset/patches/patches/patch_000002.png'), PosixPath('/kaggle/input/bboxdataset/patches/patches/patch_000003.png'), PosixPath('/kaggle/input/bboxdataset/patches/patches/patch_000004.png')]


In [20]:
# Quick schema summary
n_rows = len(df_p)
cats_present = sorted(df_p["category_id"].unique().tolist())
print(f"Patches in JSON: {n_rows}")
print(f"Unique parent image_ids: {df_p['image_id'].nunique()}")
print(f"Categories present (ids): {cats_present[:20]}{' ...' if len(cats_present)>20 else ''}")


Patches in JSON: 6000
Unique parent image_ids: 166
Categories present (ids): [0, 1, 2, 3, 6, 7, 16, 17, 18, 20]


In [21]:
# Verify files exist
missing_files = [fn for fn in df_p["file_name"] if not (PATCHES_DIR / fn).exists()]
print(f"Missing patch files on disk: {len(missing_files)}")
if missing_files[:5]: print("Examples:", missing_files[:5])

df_p.head()

Missing patch files on disk: 6000
Examples: ['patch_000000.png', 'patch_000001.png', 'patch_000002.png', 'patch_000003.png', 'patch_000004.png']


,file_name,image_id,category_id,label,bbox,fname_lower,abs_path
0,patch_000000.png,1716,20,1,"[81.32204666666667, 87.37940666666668, 121.255...",patch_000000.png,/kaggle/input/bboxdataset/patches/patches/patc...
1,patch_000001.png,2327,17,1,"[110.97295999999994, 102.15242666666666, 61.34...",patch_000001.png,/kaggle/input/bboxdataset/patches/patches/patc...
2,patch_000002.png,2349,17,1,"[117.98450000000003, 108.71353999999997, 30.00...",patch_000002.png,/kaggle/input/bboxdataset/patches/patches/patc...
3,patch_000003.png,3472,17,1,"[32.28112336666666, 73.93755159459465, 126.587...",patch_000003.png,/kaggle/input/bboxdataset/patches/patches/patc...
4,patch_000004.png,531,7,1,"[121.5694666666667, 103.33581333333336, 15.292...",patch_000004.png,/kaggle/input/bboxdataset/patches/patches/patc...


## Parent-aware, stratified train/val split

To avoid leakage, all patches from the same parent image_id stay in one split. We approximate stratification by grouping parents by their majority class and splitting parents with StratifiedKFold, yielding an 80/10/10 split. Each split has a roughly similar class distribution.

In [ ]:
# ---------- Majority class per parent (for stratification with safe fallbacks) ----------
import numpy as np
from sklearn.model_selection import train_test_split

# Build majority class per parent
maj_per_parent = (
    df_p.groupby(["image_id","category_id"])
        .size().reset_index(name="n")
        .sort_values(["image_id","n"], ascending=[True, False])
        .drop_duplicates("image_id")[["image_id","category_id"]]
        .rename(columns={"category_id":"maj_cat"})
)

# Aligned arrays: parents and their majority-class labels
maj = maj_per_parent.set_index("image_id")["maj_cat"]
parents = maj.index.to_numpy()
y = maj.values

SEED = 0

# 1) Hold out TEST = 10% (grouped by parent, try stratified by majority class, else random)
try:
    pid_trainval, pid_test = train_test_split(
        parents, test_size=0.10, random_state=SEED, stratify=y
    )
except ValueError as e:
    print("⚠️ Stratified TEST split failed:", e)
    print("➡️ Falling back to RANDOM TEST split (still grouped by parent).")
    pid_trainval, pid_test = train_test_split(
        parents, test_size=0.10, random_state=SEED, shuffle=True
    )

# 2) From remaining 90%, carve out VAL so that VAL ≈ 10% of total (i.e., 11.111% of trainval)
y_trainval = maj.loc[pid_trainval].values
try:
    pid_train, pid_val = train_test_split(
        pid_trainval, test_size=0.111111, random_state=SEED, stratify=y_trainval
    )  # 0.111111 of 90% ≈ 10%
except ValueError as e:
    print("Stratified VAL split failed:", e)
    print("Falling back to RANDOM VAL split (still grouped by parent).")
    pid_train, pid_val = train_test_split(
        pid_trainval, test_size=0.111111, random_state=SEED, shuffle=True
    )

# Safety checks
pid_train, pid_val, pid_test = set(pid_train), set(pid_val), set(pid_test)
assert pid_train.isdisjoint(pid_val) and pid_train.isdisjoint(pid_test) and pid_val.isdisjoint(pid_test), "Leakage!"

# Map back to rows
def _assign(pid):
    if pid in pid_train: return "train"
    if pid in pid_val:   return "val"
    return "test"

df_p["split"] = df_p["image_id"].map(_assign)

# Report
print(df_p["split"].value_counts())
print("unique images per split:", df_p.groupby("split")["file_name"].nunique().to_dict())


⚠️ Stratified TEST split failed: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.
➡️ Falling back to RANDOM TEST split (still grouped by parent).
⚠️ Stratified VAL split failed: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.
➡️ Falling back to RANDOM VAL split (still grouped by parent).
split
train    4756
test      630
val       614
Name: count, dtype: int64
unique images per split: {'test': 630, 'train': 4756, 'val': 614}


## Label-space mapping and class names

YOLO expects contiguous class indices 0..K-1. We therefore remap the dataset’s category_id values to [0..K-1] using the classes actually present in the patches. We provide generic names for the cat_by_id.

In [23]:
# Use existing cat names if available (from your earlier COCO EDA); else fallback.
def get_cat_name(cid):
    try:
        return cat_by_id[cid]["name"]  # defined earlier in your EDA notebook
    except Exception:
        return f"class_{cid}"

present_cids = sorted(df_p["category_id"].unique().tolist())
cid2idx = {cid: i for i, cid in enumerate(present_cids)}
idx2name = [get_cat_name(cid) for cid in present_cids]

df_p["cls_idx"] = df_p["category_id"].map(cid2idx)

print("Contiguous mapping cid→idx:", cid2idx)
print("YOLO names:", idx2name)


Contiguous mapping cid→idx: {0: 0, 1: 1, 2: 2, 3: 3, 6: 4, 7: 5, 16: 6, 17: 7, 18: 8, 20: 9}
YOLO names: ['class_0', 'class_1', 'class_2', 'class_3', 'class_6', 'class_7', 'class_16', 'class_17', 'class_18', 'class_20']


## Zero-copy YOLO dataset in a temporary workspace

We construct a YOLO-readable dataset inside the temporary workspace, no permanent copies. Each image is symlinked (or hardlinked, or copied only if linking fails) into WORKDIR/images/{train,val}. We then write YOLO label .txt files and a minimal dataset.yaml. Everything remains under WORKDIR and can be deleted automatically when the notebook ends.

In [24]:
# Create YOLO layout 
IMAGES_TRAIN = WORKDIR / "images" / "train"
IMAGES_VAL   = WORKDIR / "images" / "val"
IMAGES_TEST  = WORKDIR / "images" / "test"

LABELS_TRAIN = WORKDIR / "labels" / "train"
LABELS_VAL   = WORKDIR / "labels" / "val"
LABELS_TEST  = WORKDIR / "labels" / "test"

for d in [IMAGES_TRAIN, IMAGES_VAL, IMAGES_TEST, LABELS_TRAIN, LABELS_VAL, LABELS_TEST]:
    d.mkdir(parents=True, exist_ok=True)

In [25]:
def coco_xywh_to_yolo(x, y, w, h, W, H):
    x = max(0.0, min(float(x), W)); y = max(0.0, min(float(y), H))
    w = max(0.0, min(float(w), W - x)); h = max(0.0, min(float(h), H - y))
    cx = (x + w/2.0) / max(W, 1); cy = (y + h/2.0) / max(H, 1)
    return cx, cy, w/max(W, 1), h/max(H, 1)

In [ ]:
# This simply writes each object annotation as: One line per bounding box. Empty label files for negative images are allowed.
def write_label_txt(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for c, cx, cy, ww, hh in rows:
            f.write(f"{int(c)} {cx:.6f} {cy:.6f} {ww:.6f} {hh:.6f}\n")

In [27]:
def link_or_copy(src: Path, dst: Path):
    if dst.exists(): return "exists"
    try:
        os.symlink(src, dst); return "symlink"
    except Exception:
        try:
            os.link(src, dst); return "hardlink"
        except Exception:
            shutil.copy2(src, dst); return "copy"

In [28]:
# map split → (img_dir, lbl_dir)
split_dirs = {
    "train": (IMAGES_TRAIN, LABELS_TRAIN),
    "val":   (IMAGES_VAL,   LABELS_VAL),
    "test":  (IMAGES_TEST,  LABELS_TEST),
}

In [29]:
stats = {"missing": 0, "train_imgs": 0, "val_imgs": 0, "test_imgs": 0,
         "labels": 0, "symlink": 0, "hardlink": 0, "copy": 0}

for split, (img_dir, lbl_dir) in split_dirs.items():
    subset = df_p[df_p["split"] == split]
    for fn, grp in subset.groupby("file_name"):
        src = Path(grp["abs_path"].iloc[0])
        if not src.exists():
            stats["missing"] += 1
            continue

        dst_img = img_dir / src.name
        how = link_or_copy(src, dst_img)
        if how in stats: stats[how] += 1

        with Image.open(src) as im:
            W, H = im.size

        rows = []
        for _, r in grp.iterrows():
            bbox = r["bbox"]
            if not bbox:
                continue
            x, y, w, h = bbox
            cx, cy, ww, hh = coco_xywh_to_yolo(x, y, w, h, W, H)
            rows.append((r["cls_idx"], cx, cy, ww, hh))

        write_label_txt(lbl_dir / (dst_img.stem + ".txt"), rows)
        stats["labels"] += 1

    stats[f"{split}_imgs"] = sum(1 for _ in img_dir.glob("*.*"))

print("Export stats:", stats)

Export stats: {'missing': 0, 'train_imgs': 4756, 'val_imgs': 614, 'test_imgs': 630, 'labels': 6000, 'symlink': 6000, 'hardlink': 0, 'copy': 0}


In [30]:
# names list (idx2name) assumed already built like before
(DATA_YAML := (WORKDIR / "dataset.yaml")).write_text(
    f"""path: {WORKDIR.as_posix()}
train: images/train
val:   images/val
test:  images/test
names:
""" + "\n".join(f"  {i}: {n}" for i, n in enumerate(idx2name))
)
print("dataset.yaml →", DATA_YAML)

dataset.yaml → /kaggle/working/yolo_patches/dataset.yaml


## Train two YOLOv8 runs at different resolutions (640 & 896)

We run two trainings at imgsz=640 and imgsz=896 using the same model capacity (yolov8s.pt). All run artifacts (weights, metrics, PR curves) stay inside the temporary WORKDIR/runs/… folder.

In [31]:
device = 0 if (os.environ.get("CUDA_VISIBLE_DEVICES","") or shutil.which("nvidia-smi")) else "cpu"

# Train at 640
model_640 = YOLO("yolov8s.pt")
res_640 = model_640.train(
    data=str(DATA_YAML),
    imgsz=640,
    epochs=50,
    batch=16,
    workers=4,
    project=str(WORKDIR / "runs"),
    name="y8s_640",
    pretrained=True,
    device=device,
)

# Train at 896
model_896 = YOLO("yolov8s.pt")
res_896 = model_896.train(
    data=str(DATA_YAML),
    imgsz=896,
    epochs=50,
    batch=16,
    workers=4,
    project=str(WORKDIR / "runs"),
    name="y8s_896",
    pretrained=True,
    device=device,
)

Ultralytics 8.3.230 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yolo_patches/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=y8s_640, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=

Both models reach similar mAP50 ≈ 0.24 across runs, indicating comparable ability to detect objects under the IoU=0.5 threshold.

However:

- mAP50-95 (localization quality) is low across all settings (≈ 0.09–0.12), showing that the dataset is challenging and YOLO struggles with precise box localization.

- Recall is consistently higher at 640 (≈0.30–0.31), meaning the 640 model finds more objects overall.

- Precision tends to be higher at 896, meaning the 896 model makes fewer false positives, but also misses more instances.

- Some classes remain extremely hard, especially those with few samples (e.g., class_2, class_20), showing 0–0.02 AP in most runs.

## Validation, macro metrics, and qualitative overlays

We validate both runs on the same held-out split, aggregate macro AP across classes, and show a few qualitative predictions. Ultralytics writes per-class CSVs; we parse them if present.

In [ ]:
# REPLACE your load_per_class_df + summarize_run with these

def load_per_class_df(folder: Path):
    """
    Try to load a per-class metrics CSV from the *evaluation output folder*
    Ultralytics just created (e.g., runs/detect/y8s_640_val).
    """
    # Try common filenames across Ultralytics versions
    candidates = [
        folder / "results_per_class.csv",
        folder / "results.csv",  # sometimes includes per-class columns
    ]
    for c in candidates:
        if c.exists():
            try:
                return pd.read_csv(c)
            except Exception:
                pass
    return None


def summarize_run(tag: str, split: str = "val"):
    """
    Load best.pt for run 'tag' ('y8s_640') and evaluate on 'val' or 'test'.
    Saves results under runs/detect/<tag>_<split> and prints summary metrics.
    """
    # NOTE: Ultralytics default run path includes 'detect'
    run = WORKDIR / "runs" / tag
    best = run / "weights" / "best.pt"
    assert best.exists(), f"best.pt not found at {best}"

    # infer imgsz from tag suffix, fallback to 640
    try:
        imgsz = int(tag.split("_")[-1])
    except Exception:
        imgsz = 640

    model = YOLO(str(best))
    metrics = model.val(
        data=str(DATA_YAML),
        split=split,          # <-- 'val' or 'test'
        imgsz=imgsz,
        batch=16,
        workers=4,
        project=str(WORKDIR / "runs"),   
        name=f"{tag}_{split}",           
        device=device,
    )

    # evaluation folder Ultralytics just wrote into
    eval_dir = Path(metrics.save_dir)

    # print macro metrics from in-memory object
    print(f"[{tag} | {split.upper()}] "
          f"mAP50-95={metrics.box.map:.3f}  mAP50={metrics.box.map50:.3f}  "
          f"P={metrics.box.mp:.3f}  R={metrics.box.mr:.3f}")

    # optional per-class table (if CSV present)
    df_eval = load_per_class_df(eval_dir)
    if df_eval is not None:
        df = df_eval.rename(columns={
            "ap50": "AP@0.5", "ap50_95": "AP@0.5:0.95",
            "ap": "AP@0.5:0.95", "ap_50_95": "AP@0.5:0.95"
        })
        macro_50   = df["AP@0.5"].mean()      if "AP@0.5" in df else np.nan
        macro_5095 = df["AP@0.5:0.95"].mean() if "AP@0.5:0.95" in df else np.nan
        print(f"[{tag} | {split.upper()}] Macro AP@0.5={macro_50:.3f} | Macro AP@[.5:.95]={macro_5095:.3f}")
        if "class" in df and {"AP@0.5","AP@0.5:0.95"}.issubset(df.columns):
            display(df[["class","AP@0.5","AP@0.5:0.95"]].head())
        else:
            display(df.head())
    else:
        print(f"[{tag} | {split.upper()}] Per-class CSV not found; see plots under: {eval_dir}")


In [ ]:
# VAL evaluations
summarize_run("y8s_640", split="val")
summarize_run("y8s_896", split="val")

# TEST evaluations
summarize_run("y8s_640", split="test")
summarize_run("y8s_896", split="test")

Ultralytics 8.3.230 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 11,129,454 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 68.6±18.2 MB/s, size: 121.6 KB)
val: Scanning /kaggle/working/yolo_patches/labels/val.cache... 614 images, 322 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 614/614 1.4Mit/s 0.0s0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 39/39 5.0it/s 7.7s0.2s
                   all        614        292      0.246        0.3       0.24      0.116
               class_1         47         47      0.187      0.362      0.178     0.0718
               class_2         30         30     0.0173     0.0333     0.0355     0.0119
               class_3         31         31      0.577      0.516      0.492      0.168
               class_6         21         21      0.213      0.333       0.34      0.196
               class_7         3

In [ ]:
def find_run_dir(tag: str) -> Path:
    candidates = [
        WORKDIR / "runs" / tag,                  
        WORKDIR / "runs" / "detect" / tag        
    ]
    for d in candidates:
        if (d / "weights" / "best.pt").exists():
            return d
    raise FileNotFoundError(f"best.pt not found under any known path for tag '{tag}'. Tried: " +
                            ", ".join(str(c / 'weights' / 'best.pt') for c in candidates))

best_tag = "y8s_640"  
best_dir = find_run_dir(best_tag)
best_weights = best_dir / "weights" / "best.pt"
model_best = YOLO(str(best_weights))


TEST_DIR = WORKDIR / "images" / "test"
test_imgs = sorted(TEST_DIR.glob("*.*"))[:12]

pred_out = WORKDIR / "preds" / best_tag
pred_out.mkdir(parents=True, exist_ok=True)

_ = model_best.predict(
    source=[str(p) for p in test_imgs],
    imgsz=int(best_tag.split("_")[-1]),
    conf=0.25,
    iou=0.6,
    project=str(pred_out),
    name="demo",
    save=True
)
print("Saved overlays under:", pred_out / "demo")


In [ ]:
def find_run_dir(tag: str) -> Path:
    candidates = [
        WORKDIR / "runs" / tag,                  
        WORKDIR / "runs" / "detect" / tag        
    ]
    for d in candidates:
        if (d / "weights" / "best.pt").exists():
            return d
    raise FileNotFoundError(f"best.pt not found under any known path for tag '{tag}'. Tried: " +
                            ", ".join(str(c / 'weights' / 'best.pt') for c in candidates))

best_tag = "y8s_896" 
best_dir = find_run_dir(best_tag)
best_weights = best_dir / "weights" / "best.pt"
model_best = YOLO(str(best_weights))


TEST_DIR = WORKDIR / "images" / "test"
test_imgs = sorted(TEST_DIR.glob("*.*"))[:12]

pred_out = WORKDIR / "preds" / best_tag
pred_out.mkdir(parents=True, exist_ok=True)

_ = model_best.predict(
    source=[str(p) for p in test_imgs],
    imgsz=int(best_tag.split("_")[-1]),
    conf=0.25,
    iou=0.6,
    project=str(pred_out),
    name="demo",
    save=True
)
print("Saved overlays under:", pred_out / "demo")



0: 896x896 1 class_3, 21.1ms
1: 896x896 (no detections), 21.1ms
2: 896x896 1 class_6, 21.1ms
3: 896x896 (no detections), 21.1ms
4: 896x896 1 class_2, 21.1ms
5: 896x896 3 class_3s, 21.1ms
6: 896x896 1 class_16, 1 class_18, 21.1ms
7: 896x896 1 class_18, 21.1ms
8: 896x896 1 class_17, 21.1ms
9: 896x896 1 class_6, 21.1ms
10: 896x896 1 class_7, 1 class_20, 21.1ms
11: 896x896 (no detections), 21.1ms
Speed: 4.9ms preprocess, 21.1ms inference, 0.7ms postprocess per image at shape (1, 3, 896, 896)
Results saved to /kaggle/working/yolo_patches/preds/y8s_896/demo
Saved overlays under: /kaggle/working/yolo_patches/preds/y8s_896/demo


## Evaluate best.pt on TEST for both models

In [42]:
# Evaluate best.pt for both runs on the TEST split
RUNS = WORKDIR / "runs"

best_640 = YOLO(RUNS / "y8s_640" / "weights" / "best.pt")
best_896 = YOLO(RUNS / "y8s_896" / "weights" / "best.pt")

m640 = best_640.val(data=str(DATA_YAML), split="test")
m896 = best_896.val(data=str(DATA_YAML), split="test")

print("[640] mAP50-95:", m640.box.map, " mAP50:", m640.box.map50, " P:", m640.box.mp, " R:", m640.box.mr)
print("[896] mAP50-95:", m896.box.map, " mAP50:", m896.box.map50, " P:", m896.box.mp, " R:", m896.box.mr)


Ultralytics 8.3.230 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 11,129,454 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 97.8±22.1 MB/s, size: 131.3 KB)
val: Scanning /kaggle/working/yolo_patches/labels/test.cache... 630 images, 326 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 630/630 967.6Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 40/40 4.9it/s 8.1s0.2s
                   all        630        304      0.254      0.313       0.24      0.104
               class_1         23         23          0          0     0.0066    0.00194
               class_2         10         10     0.0969        0.3       0.15     0.0593
               class_3         26         26      0.402      0.538      0.487      0.281
               class_6         40         40      0.204      0.475      0.175      0.052
              class_16         

## Prediction TEST for both Models

In [47]:
# Save predictions for visual inspection (both models)
TEST_DIR = WORKDIR / "images" / "test"

preds_640 = best_640.predict(
    source=str(TEST_DIR), imgsz=640, conf=0.25,
    save=True, project=str(RUNS), name="pred_test_640"
)
preds_896 = best_896.predict(
    source=str(TEST_DIR), imgsz=896, conf=0.25,
    save=True, project=str(RUNS), name="pred_test_896"
)

print("Saved 640 preds to:", preds_640[0].save_dir)
print("Saved 896 preds to:", preds_896[0].save_dir)


image 1/630 /kaggle/working/yolo_patches/images/test/patch_000012.png: 640x640 1 class_1, 3 class_3s, 8.5ms
image 2/630 /kaggle/working/yolo_patches/images/test/patch_000015.png: 640x640 1 class_17, 8.0ms
image 3/630 /kaggle/working/yolo_patches/images/test/patch_000027.png: 640x640 2 class_6s, 7.8ms
image 4/630 /kaggle/working/yolo_patches/images/test/patch_000051.png: 640x640 1 class_17, 7.9ms
image 5/630 /kaggle/working/yolo_patches/images/test/patch_000085.png: 640x640 1 class_16, 7.7ms
image 6/630 /kaggle/working/yolo_patches/images/test/patch_000086.png: 640x640 3 class_3s, 7.7ms
image 7/630 /kaggle/working/yolo_patches/images/test/patch_000096.png: 640x640 1 class_18, 7.6ms
image 8/630 /kaggle/working/yolo_patches/images/test/patch_000105.png: 640x640 1 class_18, 7.8ms
image 9/630 /kaggle/working/yolo_patches/images/test/patch_000123.png: 640x640 1 class_17, 7.8ms
image 10/630 /kaggle/working/yolo_patches/images/test/patch_000128.png: 640x640 3 class_6s, 7.7ms
image 11/630 /kag

Across all baseline evaluations, the YOLOv8-small model at 640×640 (y8s_640) consistently outperforms the 896×896 version (y8s_896). On the TEST split, y8s_640 achieves mAP@0.5=0.24 and mAP@[0.5:0.95]=0.104, clearly higher than y8s_896 (0.195 and 0.089). Qualitative predictions confirm this: y8s_640 produces more detections with better localization, while y8s_896 misses objects in many patches. For all subsequent analysis, we therefore retain y8s_640 as the baseline model.


## “mIoU-style” evaluation for detection

In [ ]:
# Compute mean IoU (of true positives) on TEST for a chosen model
def iou_of(a, b):
    xa1, ya1, xa2, ya2 = a
    xb1, yb1, xb2, yb2 = b
    inter_w = max(0.0, min(xa2, xb2) - max(xa1, xb1))
    inter_h = max(0.0, min(ya2, yb2) - max(ya1, yb1))
    inter = inter_w * inter_h
    area_a = max(0.0, (xa2 - xa1)) * max(0.0, (ya2 - ya1))
    area_b = max(0.0, (xb2 - xb1)) * max(0.0, (yb2 - yb1))
    union = area_a + area_b - inter + 1e-9
    return inter / union

def load_yolo_labels(txt_path):
    boxes, classes = [], []
    if not txt_path.exists():
        return np.zeros((0,4)), np.zeros((0,), dtype=int)
    for line in txt_path.read_text().strip().splitlines():
        parts = line.strip().split()
        if len(parts) != 5: 
            continue
        c, cx, cy, w, h = map(float, parts)
        classes.append(int(c))
        boxes.append([cx, cy, w, h])
    return np.array(boxes, dtype=float), np.array(classes, dtype=int)

def yolo_to_xyxy(norm_boxes, W, H):
    if len(norm_boxes) == 0:
        return np.zeros((0,4))
    cx, cy, w, h = norm_boxes[:,0], norm_boxes[:,1], norm_boxes[:,2], norm_boxes[:,3]
    x1 = (cx - w/2) * W
    y1 = (cy - h/2) * H
    x2 = (cx + w/2) * W
    y2 = (cy + h/2) * H
    return np.stack([x1,y1,x2,y2], axis=1)

def find_best(tag: str):
    """Return Path to best.pt whether it’s under /runs or /runs/detect."""
    candidates = [
        WORKDIR / "runs" / tag / "weights" / "best.pt",
        WORKDIR / "runs" / "detect" / tag / "weights" / "best.pt",
    ]
    for p in candidates:
        if p.exists():
            print("✅ Found best.pt at:", p)
            return p
    raise FileNotFoundError(f"No best.pt found for {tag}. Tried:\n" + "\n".join(str(c) for c in candidates))

tag = "y8s_896"  

best_path = find_best(tag)
model = YOLO(str(best_path))

TEST_IMG_DIR = WORKDIR / "images" / "test"
TEST_LBL_DIR = WORKDIR / "labels" / "test"

# Run predictions 
preds = model.predict(
    source=str(TEST_IMG_DIR),
    imgsz=int(tag.split("_")[-1]),
    conf=0.25,
    iou=0.5,
    save=False,
    verbose=False
)

# --------------------------------
# Compute mean IoU per class
all_ious, per_class_ious = [], {}

for r in preds:
    im_path = Path(r.path)
    W, H = r.orig_shape[1], r.orig_shape[0]

    # Ground truth
    gt_txt = TEST_LBL_DIR / (im_path.stem + ".txt")
    gt_norm, gt_cls = load_yolo_labels(gt_txt)
    gt_xyxy = yolo_to_xyxy(gt_norm, W, H)

    # Predictions
    if r.boxes is None or len(r.boxes) == 0:
        continue
    p_xyxy = r.boxes.xyxy.cpu().numpy()
    p_cls = r.boxes.cls.cpu().numpy().astype(int)
    p_conf = r.boxes.conf.cpu().numpy()

    used_gt = set()
    order = np.argsort(-p_conf)  # sort by confidence desc
    for idx in order:
        pc = p_cls[idx]
        pb = p_xyxy[idx]
        # candidate GTs of same class not yet matched
        candidates = [j for j, gc in enumerate(gt_cls) if gc == pc and j not in used_gt]
        if not candidates:
            continue
        ious = [iou_of(pb, gt_xyxy[j]) for j in candidates]
        j_best = candidates[int(np.argmax(ious))]
        iou = ious[int(np.argmax(ious))]
        if iou >= 0.5:  # count as true positive
            used_gt.add(j_best)
            all_ious.append(iou)
            per_class_ious.setdefault(pc, []).append(iou)

# --------------------------------
def mean_or_nan(lst): 
    return float(np.mean(lst)) if lst else float("nan")

print(f"[{tag}] TP mean IoU@matched (≥0.5): {mean_or_nan(all_ious):.3f}  (n={len(all_ious)})")
for c, vals in sorted(per_class_ious.items()):
    print(f"  class {c}: mIoU={mean_or_nan(vals):.3f}  n={len(vals)}")


✅ Found best.pt at: /kaggle/working/yolo_patches/runs/y8s_896/weights/best.pt
[y8s_896] TP mean IoU@matched (≥0.5): 0.730  (n=77)
  class 1: mIoU=0.610  n=2
  class 3: mIoU=0.702  n=4
  class 4: mIoU=0.680  n=8
  class 6: mIoU=0.744  n=18
  class 7: mIoU=0.762  n=30
  class 8: mIoU=0.660  n=9
  class 9: mIoU=0.759  n=6


To better understand the localization quality of YOLOv8, we computed the mean IoU over all true positive matches (IoU ≥ 0.5) on the test set. The y8s_896 model achieves a mean IoU of 0.73 across 77 true positives, indicating that when the detector identifies an object, the predicted bounding box aligns well with the ground truth. Per-class IoU values range from 0.61 to 0.76, with the most frequent classes (e.g., class_7 and class_6) achieving the strongest box alignment. These results show that localization quality is not the primary limitation of the model. Instead, the low overall mAP is driven by limited recall: the detector successfully localizes objects it finds, but it fails to detect a large portion of instances, especially for rare classes. Therefore, the bottleneck is detection coverage rather than bounding-box accuracy.
